## Momentum Persistance Test

### Research Ques: Do winners continue to outperform?

In [ ]:
import qnt.data as qndata
import numpy as np
import xarray as xr

data = qndata.cryptodaily_load_data(
    min_date="2015-01-01"
)

close = data.sel(field="close")
is_liquid = data.sel(field="is_liquid")

close = close.where(is_liquid == 1)

mom21 = close / close.shift(time=21) - 1

future21 = close.shift(time=-21) / close - 1

corr = xr.corr(
    mom21,
    future21,
    dim="time"
)

print("Momentum Persistence")
print(float(corr.mean()))

Momentum Persistence
0.021193217811877513

## Mean Reversion Test

### Research Ques: Do short-term losers rebound

In [ ]:
mom5 = close / close.shift(time=5) - 1

future5 = close.shift(time=-5) / close - 1

corr = xr.corr(
    mom5,
    future5,
    dim="time"
)

print("Mean Reversion Test")
print(float(corr.mean()))

Mean Reversion Test
-0.023646254661119682

## Regime Impact

### Research Ques: Do bull and bear markets behave differently?

In [ ]:
btc = close.sel(asset="BTC")

btc_sma200 = btc.rolling(time=200).mean()

bull_market = btc > btc_sma200

ret = close / close.shift(time=1) - 1

ret = ret.where(is_liquid == 1)

# remove inf values
ret = ret.where(np.isfinite(ret))

bull_ret = ret.where(bull_market)

bear_ret = ret.where(~bull_market)

print("Bull Market Average Return")
print(float(bull_ret.mean()))

print()

print("Bear Market Average Return")
print(float(bear_ret.mean()))

print()

print("Bull Market Volatility")
print(float(bull_ret.std()))

print()

print("Bear Market Volatility")
print(float(bear_ret.std()))

Bull Market Average Return
0.005030391062593629

Bear Market Average Return
-0.0019902694458312905

Bull Market Volatility
0.06421155819317155

Bear Market Volatility
0.053624366190387275

## Volume Leadership
### Research Ques: Do volume spikes predict future returns?

In [ ]:
vol = data.sel(field="vol")

vol_ma20 = vol.rolling(
    time=20
).mean()

vol_ratio = vol / vol_ma20

future5 = close.shift(time=-5) / close - 1

corr = xr.corr(
    vol_ratio,
    future5,
    dim="time"
)

print("Volume Leadership")
print(float(corr.mean()))

Volume Leadership
0.04808020533358951

## Volatility Compression
### Research Ques: Does low volatility precede large moves?

In [ ]:
ret = close / close.shift(time=1) - 1

vol20 = ret.rolling(
    time=20
).std()

vol60 = ret.rolling(
    time=60
).std()

compression = vol20 / vol60

future_mag = abs(
    close.shift(time=-10) / close - 1
)

corr = xr.corr(
    compression,
    future_mag,
    dim="time"
)

print("Volatility Compression")
print(float(corr.mean()))

Volatility Compression
0.024185209702326634

## Correlation in bull vs bear regimes

In [ ]:
# Does diversification break down exactly when you need it?
ret = close / close.shift(time=1) - 1
ret = ret.where(is_liquid == 1)

btc = close.sel(asset="BTC")
btc_sma200 = btc.rolling(time=200).mean()
bull_market = btc > btc_sma200

ret_bull = ret.where(bull_market).to_pandas()
ret_bear = ret.where(~bull_market).to_pandas()

corr_bull = ret_bull.corr().mean().mean()
corr_bear = ret_bear.corr().mean().mean()

print(f"Average pairwise correlation — Bull market: {corr_bull:.3f}")
print(f"Average pairwise correlation — Bear market: {corr_bear:.3f}")
# Expect bear >> bull — diversification fails when markets crash

Average pairwise correlation — Bull market: 0.469
Average pairwise correlation — Bear market: 0.619

## Return autocorrelation (time-series momentum vs reversion)

In [ ]:
# Is each asset trend-following or mean-reverting at the single-asset level?
ret = close / close.shift(time=1) - 1
ret = ret.where(is_liquid == 1)

for lag in [1, 5, 21]:
    lagged = ret.shift(time=lag)
    ac = float(xr.corr(ret, lagged, dim="time").mean())
    direction = "trend" if ac > 0 else "mean-revert"
    print(f"Lag {lag:>2}d autocorrelation: {ac:+.4f}  ({direction})")

# Positive lag-21 = momentum persists
# Negative lag-1  = short-term noise reversal

Lag  1d autocorrelation: -0.0187  (mean-revert)
Lag  5d autocorrelation: +0.0222  (trend)
Lag 21d autocorrelation: -0.0144  (mean-revert)

## Fat tail quantification

In [ ]:
# How often do extreme moves actually happen?
import scipy.stats as stats

ret = close / close.shift(time=1) - 1
ret = ret.where(is_liquid == 1)
r = ret.values.flatten()
r = r[np.isfinite(r)]

sigma = np.std(r)
for threshold in [2, 3, 4, 5]:
    actual_pct  = np.mean(np.abs(r) > threshold * sigma) * 100
    normal_pct  = (1 - stats.norm.cdf(threshold)) * 2 * 100
    ratio       = actual_pct / normal_pct if normal_pct > 0 else float('inf')
    print(f"{threshold}σ moves: actual {actual_pct:.2f}%  |  normal dist {normal_pct:.4f}%  |  {ratio:.0f}× more frequent")

# This is what motivates inverse-vol sizing and the is_liquid hard mask

2σ moves: actual 4.34%  |  normal dist 4.5500%  |  1× more frequent
3σ moves: actual 1.50%  |  normal dist 0.2700%  |  6× more frequent
4σ moves: actual 0.68%  |  normal dist 0.0063%  |  107× more frequent
5σ moves: actual 0.36%  |  normal dist 0.0001%  |  6281× more frequent